# 05 - Utility Scoring V2

**Goal**: Implement a new, self-contained utility scoring system for the Artemis VLM Router.

**Stages**:
1. **Stage 1**: Define and compute `utility_accuracy` (combining static metrics, Glider, and Judge).
2. **Stage 2**: Define and compute `utility_cheap`, `utility_fast`, `utility_balanced` (resource-aware utilities).

**Output**: Updates `vlm_responses` table with new utility columns (accuracy + resource-aware views).

**Data flow**:
1. Query `vlm_responses` joined to `vlm_evaluations` so we only score samples that passed initial filtering.
2. Stage 1 collapses the static, Glider, Judge, and semantic signals into `utility_accuracy`.
3. Stage 2 normalizes cost/latency and derives resource-sensitive utilities before writing everything back.

**Persistence**: Prior to calculations we migrate schema (if needed) and the final stage uses a temp table to bulk-update `vlm_responses`, making the notebook idempotent.


In [18]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
# === Path Setup ===
import sys
from pathlib import Path

# Notebook paths
NOTEBOOK_DIR = Path.cwd()
ARTEMIS_DIR = NOTEBOOK_DIR.parent.parent
ROOT_DIR = ARTEMIS_DIR.parent

# Add to sys.path
for p in [str(ARTEMIS_DIR), str(ROOT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"📁 ARTEMIS_DIR: {ARTEMIS_DIR}")

# Imports
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import logging
from sqlalchemy import text
from ares.db.connection import get_engine

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(name)-15s | %(message)s', datefmt='%H:%M:%S')

📁 ARTEMIS_DIR: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
# === Database Connection ===
engine = get_engine()
print("Connected to DB")

# Ensure new columns exist (Migration step)
with engine.connect() as conn:
    # Stage 1 Column
    conn.execute(text("ALTER TABLE vlm_responses ADD COLUMN IF NOT EXISTS utility_accuracy FLOAT"))
    
    # Stage 2 Columns
    conn.execute(text("ALTER TABLE vlm_responses ADD COLUMN IF NOT EXISTS cost_norm_new FLOAT"))
    conn.execute(text("ALTER TABLE vlm_responses ADD COLUMN IF NOT EXISTS lat_norm FLOAT"))
    conn.execute(text("ALTER TABLE vlm_responses ADD COLUMN IF NOT EXISTS utility_cheap FLOAT"))
    conn.execute(text("ALTER TABLE vlm_responses ADD COLUMN IF NOT EXISTS utility_fast FLOAT"))
    conn.execute(text("ALTER TABLE vlm_responses ADD COLUMN IF NOT EXISTS utility_balanced FLOAT"))
    conn.commit()

print("Schema checked/migrated.")

Connected to DB
Schema checked/migrated.


## Stage 1: Compute `utility_accuracy`

This stage merges static correctness, Glider, Judge, and semantic signals into a single accuracy score. We only operate on rows whose original response was marked `ok = true`, and we keep the normalized signals in `df` for reuse.

### Normalized signals
- `gt_norm`: prefer `score_exact_match_normalized`, fallback to `score_numeric_match`, `score_mc_letter_match`, and finally `is_correct` so the signal stays in [0, 1].
- `glider_norm`: rescales `glider_score / 5` when Glider feedback exists.
- `judge_norm`: clamps `judge_molmo_score` into [0, 10] before dividing by 10 so judge opinions map to [0, 1].
- `semantic_norm`: directly uses `semantic_f1_f1` as-is, under the assumption that it already lies in [0, 1].
- `rank_norm`: for each `sample_id`, we turn 1-based `judge_molmo_rank_group` into a [0, 1] score via \(1 - \frac{rank - 1}{max\_rank - 1}\); rankings that are missing stay `null`.

### Aggregation strategy
Every row picks a weight template depending on the best available signals:
- Judge present: `gt` weights 0.3, `judge` 0.4, `rank` 0.2, `glider` 0.05, `semantic` 0.05.
- Judge missing but Glider/Semantic present: rely on `gt` 0.4, `glider` 0.3, `semantic` 0.3 and ignore judge/rank.
- Only static GT: fall back to `gt` weight 1.0.
We then drop absent signals, renormalize the remaining weights, and compute:
\[
utility\_accuracy = \frac{\sum\_{s} w\_s \cdot signal\_s}{\sum\_{s} w\_s}
\]
If somehow no signal is available, we default to 0, but that should only happen on very dirty data.


In [21]:
# === Load Data ===

query = """
SELECT 
    r.sample_id,
    r.model_name,
    r.response_id,
    
    -- Static Metrics
    r.ok,
    r.is_correct,
    r.score_exact_match_normalized,
    r.score_numeric_match,
    r.score_mc_letter_match,
    
    -- Resource Metrics
    r.estimated_cost_usd,
    r.latency_ms,
    
    -- Evaluation Metrics
    e.glider_score,
    e.semantic_f1_f1,
    e.judge_molmo_score,
    e.judge_molmo_rank_group
    
FROM vlm_responses r
LEFT JOIN vlm_evaluations e ON r.sample_id = e.sample_id AND r.model_name = e.model_name
WHERE r.ok = true
"""

df = pd.read_sql(query, engine)
print(f"Loaded {len(df)} rows")
df.head()

Loaded 339056 rows


,sample_id,model_name,response_id,ok,is_correct,score_exact_match_normalized,score_numeric_match,score_mc_letter_match,estimated_cost_usd,latency_ms,glider_score,semantic_f1_f1,judge_molmo_score,judge_molmo_rank_group
0,vistext_899_79527c5a,qwen2_5_vl_3b,228572,True,False,0.0,NaN,NaN,0.000105,6385.0,5.0,None,4.0,3.0
1,vistext_909_826f8225,gemma_3_27b,228650,True,False,0.0,NaN,NaN,0.000103,10835.0,4.0,None,9.0,1.0
2,vistext_918_a71670d3,qwen2_5_vl_3b,228597,True,False,0.0,NaN,NaN,0.000144,2564.0,4.0,None,NaN,NaN
3,vistext_924_2eb2fc67,deepseek_ocr,228671,True,False,0.0,NaN,NaN,0.000010,419.0,0.0,None,0.0,4.0
4,clevr_40_c6329521,deepseek_ocr,2181,True,False,0.0,NaN,NaN,0.000004,165.0,4.0,None,7.0,2.0


In [22]:
# === Compute Normalized Signals ===

def compute_gt_norm(row):
    # Prefer exact match normalized
    if pd.notnull(row['score_exact_match_normalized']):
        return float(row['score_exact_match_normalized'])
    
    # Fallback to numeric or MC match
    if pd.notnull(row['score_numeric_match']):
        return float(row['score_numeric_match'])
    if pd.notnull(row['score_mc_letter_match']):
        return float(row['score_mc_letter_match'])
        
    # Fallback to basic correctness
    return 1.0 if row['is_correct'] else 0.0

df['gt_norm'] = df.apply(compute_gt_norm, axis=1)

# Glider Signal [0, 1]
df['glider_norm'] = df['glider_score'].apply(lambda x: x / 5.0 if pd.notnull(x) else None)

# Judge Signal [0, 1]
def get_judge_norm(x):
    if pd.isnull(x): return None
    # Clamp to [0, 10] to handle outliers in dirty data
    val = float(x)
    val = max(0.0, min(val, 10.0))
    return val / 10.0

df['judge_norm'] = df['judge_molmo_score'].apply(get_judge_norm)

# Semantic F1 Signal [0, 1]
df['semantic_norm'] = df['semantic_f1_f1'].astype(float)

# Rank Signal [0, 1]
# Do this group by sample_id
def compute_rank_norm(group):
    # Filter to rows with rank
    ranked_rows = group[group['judge_molmo_rank_group'].notnull()]
    if ranked_rows.empty:
        return pd.Series([None] * len(group), index=group.index)
        
    max_rank = ranked_rows['judge_molmo_rank_group'].max()
    
    def calc_rank_score(r):
        if pd.isnull(r): return None
        if max_rank == 1: return 1.0
        return 1.0 - (r - 1) / (max_rank - 1)

    return group['judge_molmo_rank_group'].apply(calc_rank_score)

df['rank_norm'] = df.groupby('sample_id', group_keys=False).apply(compute_rank_norm)

/var/folders/l8/nbcrmtqs2838_6yr979chd1m0000gn/T/ipykernel_16402/2323458918.py:52: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['rank_norm'] = df.groupby('sample_id', group_keys=False).apply(compute_rank_norm)


In [23]:
# === Compute Utility Accuracy ===

def calculate_utility_accuracy_row(row):
    signals = {
        'gt': row['gt_norm'],
        'judge': row['judge_norm'],
        'rank': row['rank_norm'],
        'glider': row['glider_norm'],
        'semantic': row['semantic_norm']
    }
    
    # Base Weights
    # Rule 1: If Judge is present, it's a strong signal
    if pd.notnull(signals['judge']):
        weights = {
            'gt': 0.3,
            'judge': 0.4,
            'rank': 0.2,
            'glider': 0.05,
            'semantic': 0.05
        }
    # Rule 2: If Judge missing, rely on Glider/Semantic + GT
    elif pd.notnull(signals['glider']) or pd.notnull(signals['semantic']):
        weights = {
            'gt': 0.4,
            'judge': 0.0, # N/A
            'rank': 0.0,  # N/A
            'glider': 0.3,
            'semantic': 0.3
        }
    # Rule 3: Only Static GT exists
    else:
        weights = {
            'gt': 1.0,
            'judge': 0.0,
            'rank': 0.0,
            'glider': 0.0,
            'semantic': 0.0
        }
        
    # Filter to available signals and renormalize weights
    available_weights = {}
    total_weight = 0.0
    score_sum = 0.0
    
    for k, w in weights.items():
        val = signals[k]
        if pd.notnull(val) and w > 0:
            available_weights[k] = w
            total_weight += w
            score_sum += val * w
            
    if total_weight == 0:
        return 0.0 # Should not happen if data is clean, but fallback
        
    return score_sum / total_weight

df['utility_accuracy'] = df.apply(calculate_utility_accuracy_row, axis=1)

# Check stats
print("Utility Accuracy Distribution:")
print(df['utility_accuracy'].describe())

Utility Accuracy Distribution:
count    339056.000000
mean          0.437965
std           0.328108
min           0.000000
25%           0.042105
50%           0.442105
75%           0.673684
max           1.000000
Name: utility_accuracy, dtype: float64


## Stage 2: Resource-Aware Utilities

After `utility_accuracy` is stable, we layer in resource signals so downstream selection can trade off cost and latency.

### Normalized cost and latency
- `cost_norm_new`: min-max normalization of `estimated_cost_usd`, mapping the cheapest response to 0 and the most expensive to 1.
- `lat_norm`: the same min-max transform on `latency_ms`, with missing or constant data collapsed to 0.

### Derived utilities
- `utility_cheap`: zero unless `utility_accuracy >= 0.3`; otherwise it is \(0.6 \cdot utility\_accuracy + 0.4 \cdot (1 - cost\_norm\_new)\).
- `utility_fast`: zero unless `utility_accuracy >= 0.3`; if `lat_norm` is null the row is marked 0 because we cannot prove slowness; otherwise it mixes \(0.6 \cdot utility\_accuracy + 0.4 \cdot (1 - lat\_norm)\).
- `utility_balanced`: always defined and blends accuracy, cost, and latency as \(0.5 \cdot utility\_accuracy + 0.25 \cdot (1 - cost\_norm\_new) + 0.25 \cdot (1 - lat\_norm)\), treating missing latency as the worst case so the latency term contributes 0.
These resource utilities give cheap/fast/balanced signals that still honor our accuracy floor while punishing expensive or slow responses.


In [24]:
# === Normalize Costs and Latency ===

# Cost Normalization [0, 1] (Lower is better, but here we just map values)
min_cost = df['estimated_cost_usd'].min()
max_cost = df['estimated_cost_usd'].max()
if max_cost > min_cost:
    df['cost_norm_new'] = (df['estimated_cost_usd'] - min_cost) / (max_cost - min_cost)
else:
    df['cost_norm_new'] = 0.0
    
# Latency Normalization [0, 1]
# Filter out None latencies for calculation, fill with max or something reasonable if needed
min_lat = df['latency_ms'].min()
max_lat = df['latency_ms'].max()
if pd.notnull(max_lat) and max_lat > min_lat:
    df['lat_norm'] = (df['latency_ms'] - min_lat) / (max_lat - min_lat)
else:
    df['lat_norm'] = 0.0 # Or handled as missing

# === Utility Cheap ===
# Goal: Penalize high cost, but require decent accuracy.
# Logic: utility_cheap = utility_balanced - (weight * cost)
# Let's use a simpler formulation: 
# If acc < 0.3 -> 0.0 (Unacceptable)
# Else -> 0.6 * Acc + 0.4 * (1 - cost_norm)
def calc_cheap(row):
    if row['utility_accuracy'] < 0.3:
        return 0.0
    c_score = 1.0 - row['cost_norm_new']
    return 0.6 * row['utility_accuracy'] + 0.4 * c_score

df['utility_cheap'] = df.apply(calc_cheap, axis=1)

# === Utility Fast ===
# Similar to cheap but with latency
def calc_fast(row):
    if row['utility_accuracy'] < 0.3:
        return 0.0
    if pd.isnull(row['lat_norm']):
        return 0.0 # Treat missing latency as bad
    l_score = 1.0 - row['lat_norm']
    return 0.6 * row['utility_accuracy'] + 0.4 * l_score

df['utility_fast'] = df.apply(calc_fast, axis=1)

# === Utility Balanced ===
# Trade-off: acc vs (cost, lat)
# Score = Acc - Penalty * (Cost + Lat)
# Or weighted average.
# Let's do: 0.5 * Acc + 0.25 * (1-Cost) + 0.25 * (1-Lat)
def calc_balanced(row):
    c_score = 1.0 - row['cost_norm_new']
    l_score = 1.0 - row['lat_norm'] if pd.notnull(row['lat_norm']) else 0.0
    return 0.5 * row['utility_accuracy'] + 0.25 * c_score + 0.25 * l_score

df['utility_balanced'] = df.apply(calc_balanced, axis=1)

print("Utilities Calculated.")
df[['model_name', 'utility_accuracy', 'utility_cheap', 'utility_fast', 'utility_balanced']].head()

Utilities Calculated.


,model_name,utility_accuracy,utility_cheap,utility_fast,utility_balanced
0,qwen2_5_vl_3b,0.291228,0.000000,0.000000,0.626251
1,gemma_3_27b,0.631579,0.773321,0.736018,0.785442
2,qwen2_5_vl_3b,0.342857,0.597824,0.595655,0.660210
3,deepseek_ocr,0.000000,0.000000,0.000000,0.498712
4,deepseek_ocr,0.336842,0.601951,0.601581,0.667997


## Stage 3: Persist Calculated Utilities

To keep the notebook idempotent we already ensured the columns exist and now bulk push the computed rows back to `vlm_responses`.
Using a temporary table (`temp_utility_updates`), we upload `response_id` plus the new metrics via `to_sql`, then run a single `UPDATE ... FROM` statement inside `engine.begin()` so the write is atomic. The temp table drops at the end of the session.


In [25]:
# === Write Back to Database ===

# Efficient bulk update using a temporary table pattern
# 1. Create temp table
# 2. Insert metrics
# 3. Update main table from temp

temp_table_name = "temp_utility_updates"

update_df = df[['response_id', 
                'utility_accuracy', 
                'cost_norm_new', 
                'lat_norm', 
                'utility_cheap', 
                'utility_fast', 
                'utility_balanced']].copy()

with engine.begin() as conn:
    # Ensure no leftover temp table
    conn.execute(text(f"DROP TABLE IF EXISTS {temp_table_name}"))
    # Create temp table
    conn.execute(text(f"""
        CREATE TEMP TABLE {temp_table_name} (
            response_id INTEGER PRIMARY KEY,
            utility_accuracy FLOAT,
            cost_norm_new FLOAT,
            lat_norm FLOAT,
            utility_cheap FLOAT,
            utility_fast FLOAT,
            utility_balanced FLOAT
        )
    """))
    
    # Upload data to temp table
    update_df.to_sql(temp_table_name, conn, if_exists='append', index=False)
    print(f"Uploaded {len(update_df)} rows to temp table")
    
    # Update main table
    update_query = text(f"""
        UPDATE vlm_responses AS r
        SET 
            utility_accuracy = t.utility_accuracy,
            cost_norm_new = t.cost_norm_new,
            lat_norm = t.lat_norm,
            utility_cheap = t.utility_cheap,
            utility_fast = t.utility_fast,
            utility_balanced = t.utility_balanced
        FROM {temp_table_name} AS t
        WHERE r.response_id = t.response_id
    """)
    
    result = conn.execute(update_query)
    print(f"Updated {result.rowcount} rows in vlm_responses")
    
    # Cleanup handled by auto-drop of temp table, but explicit drop involves transaction commit implicitly sometimes? 
    # Temp tables drop at end of session. Engine.begin() commits at exit.

print("Done.")

Uploaded 339056 rows to temp table
Updated 339056 rows in vlm_responses
Done.
